In [ ]:
import json
from tqdm import tqdm

import nlpaug.augmenter.word as naw
import numpy as np
import pandas as pd
from sklearn.model_selection import (
    train_test_split,
)
from tokenizers import BertWordPieceTokenizer


In [10]:
RANDOM_STATE = 654321
TEST_SIZE = 0.30

In [26]:
CORPUS_PATH = './echo_from_epikriz_corpus.csv'
MODEL_PATH = './rubert_cased_L-12_H-768_A-12_pt_v1'
TEXT_PATH = './train_text.csv'
OUTPUT_PATH = './'

In [35]:
data = pd.read_csv(CORPUS_PATH)

data.shape

(6729, 2)

In [36]:
data.head(2)

,echo_from_epikriz,target
0,плахов клименк увелич тонк подвижн измен дуг п...,0.0
1,лев предсерд особен увелич лев желудочек тейхо...,0.0


In [37]:
data = data.dropna()
data.isna().sum()

echo_from_epikriz    0
target               0
dtype: int64

In [15]:
df_train, df_eval = train_test_split(
    data, 
    stratify=data['target'], 
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
    )

df_train.shape, df_eval.shape,

((4709, 2), (2019, 2))

In [19]:
df_train['target'].value_counts()[0]

4407

### Train data text augmentation

In [20]:
aug = naw.ContextualWordEmbsAug(
    model_path=MODEL_PATH, 
    action='insert',
    # device='mps'
    )

In [21]:
SAMPLES_COUT = df_train['target'].value_counts()[0] - df_train['target'].value_counts()[1]
SAMPLES_COUT

4105

In [22]:
def augmentation_text(df, samples=SAMPLES_COUT, pr=0.7):
    aug.aug_p  = pr
    new_text = []

    # selecting the minority class samples
    df_n = df[df.target==1].reset_index(drop=True)

    # data augmentation loop
    for i in tqdm(np.random.randint(0, len(df_n), samples)):

            text = df_n.iloc[i]['echo_from_epikriz']
            augmented_text = aug.augment(text)
            augmented_text = ''.join(augmented_text)
            new_text.append(augmented_text)

    # dataframe
    new = pd.DataFrame({'echo_from_epikriz': new_text,'target': 1})
    df = pd.concat([df, new], ignore_index=True).reset_index(drop=True)
    df = df.sample(frac=1).reset_index(drop=True)
    return df

df_train = augmentation_text(df_train)

100%|██████████| 4105/4105 [47:25<00:00,  1.44it/s]  


In [23]:
df_train['target'].value_counts()

target
0.0    4407
1.0    4407
Name: count, dtype: int64

In [24]:
df_train.sample(5)

,echo_from_epikriz,target
1469,митральн клапа трехкомпонентн полост групп пап...,0.0
8499,гипоплазир умерен тем увелич что створк уплотн...,1.0
5800,лев отдел умерн до увелич уст измен клапа изме...,1.0
4780,N увелич м норм заканчива со брахеоцефальн сос...,1.0
8279,лев предсерд особен увелич лев желудочек тейхо...,0.0


In [39]:
df_train.to_csv(OUTPUT_PATH + 'train.csv', index=False)
data['echo_from_epikriz'].to_csv(OUTPUT_PATH + 'train_text.csv', index=False)
df_eval.to_csv(OUTPUT_PATH + 'eval.csv', index=False)


### Vocab file creation

In [40]:
tokenizer = BertWordPieceTokenizer(
    clean_text=True,
    handle_chinese_chars=False,
    strip_accents=False,
    lowercase=False,
    )

In [41]:
trainer = tokenizer.train( 
    TEXT_PATH,
    vocab_size=32000,
    min_frequency=2,
    show_progress=True,
    special_tokens=['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]'],
    limit_alphabet=1000,
    wordpieces_prefix='##'
)

In [42]:
tokenizer.save(OUTPUT_PATH +'vocab_new.json', pretty=True)

In [43]:
with open(OUTPUT_PATH + 'vocab_new.json') as f:
    data_json = json.load(f)

vocab = data_json['model']['vocab']
vocab_txt = ''
for key, value in vocab.items():
    vocab_txt += key
    vocab_txt += '\n'
    
vocab_txt = vocab_txt[:-1]
    
with open(OUTPUT_PATH + 'vocab_new.txt', 'wt') as f:
    f.write(vocab_txt)